# Adaptive QA: Stage 5 Final Holdout Experiments (100 Untouched Questions)

This notebook runs the **Stage 5** final holdout experiments on **100 untouched QANTA 2021 questions** (`200` prefix states, `400` action nodes, `1,000` logical LLM calls, `200` BM25 retrievals) strictly disjoint from all 50 Stage 3 development questions:
1. **Isolated Workspace:** `/content/drive/MyDrive/AdaptiveQA2021_consistency/adaptive-qa-stage05-holdout100` (preserves Stage 2, Stage 3, and Stage 4 untouched).
2. **Frozen Training-Only Corpus:** Reuses `evidence_corpus_qanta2021_train_top500.jsonl` (`496` training-only articles, `1,973` passages; zero pilot/dev/holdout access).
3. **Model & Prompt:** Pinned `Qwen/Qwen2.5-3B-Instruct` (`aa8e72537993ba99e69dfaafa59ed015b17504d1`) with `qa-answer-v4` (`100%` JSON validity).
4. **Full Evaluation & Figures:** Runs `verify_collection.py`, `pilot-report`, and `evaluate` (calibration ECE/Brier/AUROC/AURC, 7-policy comparison, $\lambda$ cost sweep, risk–coverage curves, and feature-group ablations) and downloads **`STAGE_05_HOLDOUT_RESULTS.zip`** (~14–15 minutes on a free Colab Tesla T4 GPU; durable response caching lets you resume automatically if disconnected).

In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
from io import BytesIO
import os, sys, json, zipfile, shutil, subprocess

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency')
PREV_STAGE4 = BASE_DIR / 'adaptive-qa-stage04-corrective'
PREV_STAGE3 = BASE_DIR / 'adaptive-qa-stage03-v3'
PREV_STAGE2 = BASE_DIR / 'adaptive-confidence-agent'
PROJECT = BASE_DIR / 'adaptive-qa-stage05-holdout100'
PROJECT.mkdir(parents=True, exist_ok=True)
print('Stage 5 isolated workspace:', PROJECT)

In [ ]:
# Load stage_05_holdout.zip automatically from Drive (or prompt for upload)
zip_cands = sorted(BASE_DIR.glob('stage_05_holdout*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_cands:
    print('Loading Stage 5 archive from Drive:', zip_cands[0])
    raw_bytes = zip_cands[0].read_bytes()
else:
    uploaded = files.upload()
    raw_bytes = next(iter(uploaded.values()))

with zipfile.ZipFile(BytesIO(raw_bytes)) as zf:
    for entry in zf.infolist():
        if entry.is_dir():
            continue
        parts = PurePosixPath(entry.filename).parts
        rel = PurePosixPath(*parts[1:]) if parts[0] == 'adaptive-qa' else PurePosixPath(*parts)
        target = PROJECT / str(rel)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(zf.read(entry.filename))

os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[gpu,data]'], check=True)
print('Installed Stage 5 package in:', PROJECT)

In [ ]:
# Copy raw QANTA 2021 files, Stage 3 pilot50 exclusion list, and frozen training-only corpus from Drive
ignore_gs = shutil.ignore_patterns('*.gsheet', '*.gdoc', '*.gslides')

for donor in (PREV_STAGE4, PREV_STAGE3, PREV_STAGE2):
    if (donor / 'data/raw/qanta2021/qanta.dev.2021.12.20.json').exists() and not (PROJECT / 'data/raw/qanta2021/qanta.dev.2021.12.20.json').exists():
        shutil.copytree(donor / 'data/raw/qanta2021', PROJECT / 'data/raw/qanta2021', dirs_exist_ok=True, ignore=ignore_gs)
        print('Copied raw QANTA 2021 files from:', donor)
    if (donor / 'data/processed/qanta2021/pilot50.jsonl').exists() and not (PROJECT / 'data/processed/qanta2021/pilot50.jsonl').exists():
        shutil.copytree(donor / 'data/processed/qanta2021', PROJECT / 'data/processed/qanta2021', dirs_exist_ok=True, ignore=ignore_gs)
        print('Copied Stage 3 pilot50 exclusion reference from:', donor)

corpus_dst = PROJECT / 'data/raw/evidence_corpus_qanta2021_train_top500.jsonl'
if not corpus_dst.exists():
    corpus_src = PREV_STAGE4 / 'data/raw/evidence_corpus_qanta2021_train_top500.jsonl'
    if corpus_src.is_file():
        corpus_dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(corpus_src, corpus_dst)
        if (PREV_STAGE4 / 'data/raw/wikipedia_train_top500').is_dir():
            shutil.copytree(PREV_STAGE4 / 'data/raw/wikipedia_train_top500', PROJECT / 'data/raw/wikipedia_train_top500', dirs_exist_ok=True, ignore=ignore_gs)
        print('Copied frozen strictly training-only Wikipedia corpus from Stage 4!')
    else:
        subprocess.run([sys.executable, '-u', 'scripts/build_wikipedia_corpus.py', '--contact', 'https://qanta-org.github.io', '--limit', '500', '--overwrite-corpus'], cwd=PROJECT, check=True)

# Run all 30 unit tests (including Stage 5 holdout disjointness test)
res = subprocess.run([sys.executable, 'scripts/run_tests.py'], cwd=PROJECT, capture_output=True, text=True)
test_log = (res.stdout or '') + (res.stderr or '')
print(test_log)
assert res.returncode == 0, 'Unit tests failed!'
(PROJECT / 'results').mkdir(parents=True, exist_ok=True)
(PROJECT / 'results/stage05_tests.txt').write_text(test_log, encoding='utf-8')

## 1. Freeze 100 Untouched Holdout Questions (Zero Overlap with Stage 3 Development Set)

In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/prepare_stage5_holdout.py', '--count', '100'], cwd=PROJECT, check=True)
mf = json.loads((PROJECT / 'data/processed/qanta2021_stage5/holdout100_manifest.json').read_text())
print(json.dumps(mf, indent=2))
assert mf['overlap_with_stage3_pilot50'] == 0

## 2. Run Stage 5 Holdout Collection on Free Colab T4 GPU (~14–15 Minutes)

Per-call responses are cached durably on Google Drive (`data/cache_stage05_holdout100/llm`), so if Colab disconnects at any point, re-running this cell automatically resumes from the cache without repeating completed calls.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Please switch Colab runtime to T4 GPU (Runtime -> Change runtime type -> T4 GPU).'
print('GPU:', torch.cuda.get_device_name(0), f'({torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB)')

CFG = 'configs/stage05_holdout100.yaml'
subprocess.run([sys.executable, '-u', 'scripts/run.py', 'prepare', '--config', CFG], cwd=PROJECT, check=True)
subprocess.run([sys.executable, '-u', 'scripts/run.py', 'estimate', '--config', CFG], cwd=PROJECT, check=True)
subprocess.run([sys.executable, '-u', 'scripts/run.py', 'collect', '--config', CFG, '--execute-model'], cwd=PROJECT, check=True)
subprocess.run([sys.executable, '-u', 'scripts/verify_collection.py', '--config', CFG], cwd=PROJECT, check=True)
subprocess.run([sys.executable, '-u', 'scripts/run.py', 'pilot-report', '--config', CFG], cwd=PROJECT, check=True)

## 3. Run Stage 5 Evaluation (Calibration, 7 Policies, $\lambda$ Cost Sweep, Risk–Coverage & Feature Ablations)

In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/run.py', 'evaluate', '--config', CFG], cwd=PROJECT, check=True)

from IPython.display import display, Image, Markdown
run_dir = PROJECT / 'results/runs/qanta2021_stage05_holdout100_qwen3b_v4'
display(Markdown((run_dir / 'pilot_report.md').read_text()))

s5 = json.loads((run_dir / 'stage5_controller_evaluation.json').read_text())
print('=== RQ1 Calibration (Holdout 100, 400 Nodes) ===')
print(json.dumps(s5['rq1_calibration_summary'], indent=2))
print('\n=== RQ3 Policy Comparison (Holdout 100, lambda=0.1, R_wrong=-1.0) ===')
for row in s5['rq3_policy_comparison_concept_r1_lambda_0_1']:
    print(f"{row['policy']:40s} | acc={row['overall_accuracy']:.3f} | sel_acc={row['selective_accuracy']:.3f} | cov={row['coverage']:.2f} | ret={row['retrieve_rate']:.2f} | tok={row['avg_total_tokens']:.1f} | U={row['mean_utility']:+.4f}")

for fig in sorted((run_dir / 'figures').glob('*.png')):
    print('Figure:', fig.name)
    display(Image(filename=str(fig)))

## 4. Package & Download `STAGE_05_HOLDOUT_RESULTS.zip`

In [ ]:
out_zip = Path('/content/STAGE_05_HOLDOUT_RESULTS.zip')
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(PROJECT / 'results/stage05_tests.txt', 'stage05_tests.txt')
    zf.write(PROJECT / 'data/processed/qanta2021_stage5/holdout100_manifest.json', 'holdout100_manifest.json')
    if (PROJECT / 'data/raw/wikipedia_train_top500/corpus_manifest.json').is_file():
        zf.write(PROJECT / 'data/raw/wikipedia_train_top500/corpus_manifest.json', 'train_top500_corpus_manifest.json')
    for fn in ('pilot_report.json', 'pilot_report.md', 'collection_verification.json', 'manifest.json', 'stage4_calibration_report.json', 'stage5_controller_evaluation.json'):
        p = run_dir / fn
        if p.is_file():
            zf.write(p, fn)
    for sub in ('tables', 'figures'):
        for p in sorted((run_dir / sub).glob('*')):
            if p.is_file() and p.suffix not in ('.gsheet', '.gdoc', '.gslides'):
                zf.write(p, f'{sub}/{p.name}')

print('Created:', out_zip, 'bytes:', out_zip.stat().st_size)
files.download(str(out_zip))